# Phase 7 - Evaluation and Result Analysis

This notebook evaluates all saved LightGBM predictions, compares aggregate and per-class metrics, and reviews minority-class behavior.

In [ ]:
# Cell 1 - Locate the project root and import evaluation dependencies.
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

project_root = next(
    candidate
    for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / 'configs' / 'config.yaml').exists()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f'Project root: {project_root}')

In [ ]:
# Cell 2 - Load shared configuration and verify Phase 6 prediction artifacts.
from src.data_loading import load_config, resolve_project_path

config_path = project_root / 'configs' / 'config.yaml'
config = load_config(config_path)
processed_dir = resolve_project_path(config['paths']['processed_dir'], project_root)
metrics_dir = resolve_project_path(config['paths']['metrics_dir'], project_root)
figures_dir = resolve_project_path(config['paths']['figures_dir'], project_root)
scenarios = config['imbalance']['scenarios']
prediction_paths = [processed_dir / f'y_pred_{scenario}.npy' for scenario in scenarios]
missing_paths = [path for path in prediction_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(f'Missing Phase 6 predictions: {missing_paths}')

print(f'Prediction artifacts ready: {len(prediction_paths)}')

In [ ]:
# Cell 3 - Run the canonical evaluation for all four scenarios.
from src.evaluation import evaluate_all_scenarios

evaluation_result = evaluate_all_scenarios(config_path)
summary = evaluation_result['summary']
display(summary)

In [ ]:
# Cell 4 - Cross-check aggregate results against independently reviewed values.
review_reference = {
    's1_none': {'accuracy': 0.8436, 'macro_f1': 0.3142, 'macro_recall': 0.3376},
    's2_class_weight': {'accuracy': 0.7285, 'macro_f1': 0.4249, 'macro_recall': 0.5620},
    's3_upsampling': {'accuracy': 0.7278, 'macro_f1': 0.4233, 'macro_recall': 0.5580},
    's4_downsampling': {'accuracy': 0.6155, 'macro_f1': 0.2923, 'macro_recall': 0.4900},
}
for scenario, expected_metrics in review_reference.items():
    actual = summary[summary['scenario'] == scenario].iloc[0]
    for metric, expected in expected_metrics.items():
        assert round(float(actual[metric]), 4) == expected, (scenario, metric)

assert summary.loc[summary['macro_f1'].idxmax(), 'scenario'] == 's2_class_weight'
assert summary.loc[summary['macro_recall'].idxmax(), 'scenario'] == 's2_class_weight'
print('Aggregate metrics match the independent review values.')

In [ ]:
# Cell 5 - Compare precision, recall, and F1 for the five minority-focus classes.
minority = evaluation_result['per_class'][
    evaluation_result['per_class']['is_minority_focus']
].copy()
minority_pivot = minority.pivot(
    index='class_name',
    columns='scenario',
    values=['precision', 'recall', 'f1_score'],
)
display(minority_pivot)

minority_means = (
    minority.groupby('scenario')[['precision', 'recall', 'f1_score']]
    .mean()
    .reindex(scenarios)
)
display(minority_means)

In [ ]:
# Cell 6 - Display the aggregate comparison and all confusion-matrix figures.
display(Image(filename=evaluation_result['metric_figure_path']))
for scenario in scenarios:
    figure_path = figures_dir / f'confusion_matrix_{scenario}.png'
    if not figure_path.exists():
        raise FileNotFoundError(figure_path)
    display(Markdown(f'## {scenario}'))
    display(Image(filename=str(figure_path)))

In [ ]:
# Cell 7 - Review the generated qualitative findings and methodological interpretation.
analysis = evaluation_result['qualitative_analysis']
for finding in analysis['key_findings']:
    print(f'- {finding}')

equivalence = analysis['s2_s3_equivalence']
print(f"S2-S3 macro F1 delta: {equivalence['macro_f1_absolute_delta']:.6f}")
print(
    'S3/S2 training-time ratio: '
    f"{equivalence['training_time_ratio_s3_over_s2']:.3f}"
)
print(equivalence['explanation'])

In [ ]:
# Cell 8 - List the Phase 7 artifacts prepared for result reporting.
artifact_paths = [
    Path(evaluation_result['summary_path']),
    Path(evaluation_result['per_class_path']),
    Path(evaluation_result['minority_path']),
    Path(evaluation_result['qualitative_path']),
    Path(evaluation_result['analysis_markdown_path']),
    Path(evaluation_result['metric_figure_path']),
]
artifact_paths.extend(
    figures_dir / f'confusion_matrix_{scenario}.png' for scenario in scenarios
)
artifact_summary = pd.DataFrame({
    'artifact': [str(path) for path in artifact_paths],
    'exists': [path.exists() for path in artifact_paths],
})
assert artifact_summary['exists'].all()
display(artifact_summary)
print('Phase 7 evaluation artifacts are complete.')